In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import openslide
from tqdm import tqdm # Giúp hiển thị thanh tiến trình cho đẹp
import numpy as np

class WSIPatchDataset(Dataset):
    def __init__(self, svs_path, coords_list, patch_size, transform=None):
        self.svs_path = svs_path
        self.coords_list = coords_list
        self.patch_size = patch_size
        self.transform = transform
        # Khởi tạo slide = None để tránh lỗi pickle khi dùng multiprocessing
        self.slide = None 

    def __len__(self):
        return len(self.coords_list)

    def __getitem__(self, idx):
        # Chỉ mở file svs khi worker process thực sự gọi hàm này
        if self.slide is None:
            self.slide = openslide.OpenSlide(self.svs_path)
            
        x, y = self.coords_list[idx]
        
        # Cắt ảnh trực tiếp từ file SVS
        patch = self.slide.read_region((x, y), 0, (self.patch_size, self.patch_size)).convert("RGB")
        
        # Áp dụng chuẩn hóa (ToTensor, Normalize...)
        if self.transform:
            patch = self.transform(patch)
            
        # Trả về tensor của ảnh và tọa độ tương ứng
        return patch, torch.tensor([x, y])

1